# 🏭 Web Scraping with Python — Notebook 3
## Production-Style Pipelines & Real-World Patterns

---

## 🎯 What This Notebook Is About

You now know `requests + BS4` and `Selenium`. In real projects, you don't just run a script once — you build a **pipeline**: a system that fetches, cleans, validates, and stores data reliably.

### Beginner Scraper vs Production Scraper

```
BEGINNER SCRAPER                  PRODUCTION SCRAPER
────────────────────────────────────────────────────────────
requests.get(url)                 Smart fetcher: try requests,
                                  fall back to Selenium if needed

soup.find(...).text               Safe extraction + cleaning
                                  + type conversion in one step

print(data)                       Save to SQLite / CSV / JSON
                                  with deduplication

Crashes on first error            Retry with backoff + logging

Scrapes the same data twice       Deduplication + checkpointing

Hard-coded URLs in code           Config-driven, reusable design
```

### What You'll Build in This Notebook

```
Cell 2   Smart Fetcher      requests → Selenium fallback
Cell 3   Reusable Cleaner   clean, type-convert in one place
Cell 4   Retry + Backoff    never crash on a flaky network
Cell 5   Logging            track everything your scraper does
Cell 6   SQLite Storage     real database, not just CSV
Cell 7   Deduplication      never scrape the same data twice
Cell 8   Full Pipeline      all pieces wired together
Cell 9   Project: Jobs      scrape job listings end-to-end
Cell 10  Project: Prices    scrape + track price changes
```

## 🤖 robots.txt — Always Check Before Scraping

Every website has a `robots.txt` file that tells scrapers what they can and cannot access.
Checking it is both **ethical** and **legally important**.

```
https://quotes.toscrape.com/robots.txt
```

```python
from urllib.robotparser import RobotFileParser

rp = RobotFileParser()
rp.set_url('https://site.com/robots.txt')
rp.read()

rp.can_fetch('*', 'https://site.com/products')   # True = allowed
rp.can_fetch('*', 'https://site.com/admin')       # False = blocked
```

> ⚠️ **Rule**: If `can_fetch()` returns `False` — **don't scrape that URL**.
> Ignoring robots.txt can lead to IP bans or legal issues.

In [ ]:
# ROBOTS.TXT CHECK — Add this at the start of every real scraper

from urllib.robotparser import RobotFileParser
from urllib.parse import urlparse

def can_scrape(base_url, target_url, user_agent='*'):
    """
    Check if robots.txt allows scraping target_url.
    Returns True if allowed, False if blocked.
    """
    parsed   = urlparse(base_url)
    robots_url = f'{parsed.scheme}://{parsed.netloc}/robots.txt'
    rp = RobotFileParser()
    rp.set_url(robots_url)
    try:
        rp.read()
        return rp.can_fetch(user_agent, target_url)
    except Exception as e:
        logger.warning(f'Could not read robots.txt: {e}. Proceeding cautiously.')
        return True   # If can't read, assume allowed but log it

# ── Test on our practice sites ──
sites = [
    ('https://quotes.toscrape.com', 'https://quotes.toscrape.com/page/1/'),
    ('https://books.toscrape.com',  'https://books.toscrape.com/catalogue/'),
    ('https://realpython.github.io', 'https://realpython.github.io/fake-jobs/'),
]

print('Robots.txt check:')
print('-' * 55)
for base, target in sites:
    allowed = can_scrape(base, target)
    status  = '✅ ALLOWED' if allowed else '❌ BLOCKED'
    print(f'  {status}  {target[:45]}')


## 🧠 Cell 2 — Smart Fetcher: requests First, Selenium Fallback

The #1 production rule: **always try requests first** — it's 10x faster and uses no memory.
Only switch to Selenium if the page is JS-rendered.

### How to Detect if a Page Needs Selenium

```python
# Fetch with requests
r = requests.get(url)
soup = BeautifulSoup(r.text, 'html.parser')

# Check if the data you expect is actually there
if soup.find('div', class_='product'):     # Data is in plain HTML -> use requests
    ...
else:
    # Data not found -> page is JS-rendered -> use Selenium
    ...
```

### The Smart Fetcher Function

```
smart_fetch(url)
      |
      v
  requests.get()
      |
      +--> check_indicator present? --Yes--> return soup (fast path)
      |                               
      +--> No? --> launch Selenium --> page_source --> return soup (slow path)
```

In [ ]:
# CELL 2 — Imports + Smart Fetcher

import requests
import time
import random
import json
import logging
from bs4 import BeautifulSoup

from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from webdriver_manager.chrome import ChromeDriverManager

HEADERS = {
    'User-Agent': (
        'Mozilla/5.0 (Windows NT 10.0; Win64; x64) '
        'AppleWebKit/537.36 Chrome/120.0.0.0 Safari/537.36'
    )
}

def get_driver(headless=True):
    opts = Options()
    if headless: opts.add_argument('--headless')
    opts.add_argument('--no-sandbox')
    opts.add_argument('--disable-dev-shm-usage')
    opts.add_argument('--disable-blink-features=AutomationControlled')
    opts.add_argument('--window-size=1920,1080')
    opts.add_argument(f'user-agent={HEADERS["User-Agent"]}')
    opts.add_experimental_option('excludeSwitches', ['enable-logging'])
    return webdriver.Chrome(service=Service(ChromeDriverManager().install()), options=opts)


def smart_fetch(url, js_indicator=None, timeout=10):
    """
    Fetches a URL smartly:
    - Tries requests first (fast, no JS)
    - Falls back to Selenium if js_indicator tag is missing

    js_indicator: a CSS selector that should be present if page loaded correctly
                  e.g. 'div.quote', '.product', '#results'
    Returns: BeautifulSoup object
    """
    # Step 1: Try requests first
    try:
        r = requests.get(url, headers=HEADERS, timeout=timeout)
        r.raise_for_status()
        soup = BeautifulSoup(r.text, 'html.parser')

        # Step 2: Check if expected content is present
        if js_indicator is None or soup.select_one(js_indicator):
            print(f'  [requests] {url}')
            return soup

        print(f'  [requests] Content missing -> falling back to Selenium')

    except requests.exceptions.RequestException as e:
        print(f'  [requests] Failed: {e} -> falling back to Selenium')

    # Step 3: Fall back to Selenium
    driver = get_driver(headless=True)
    try:
        driver.get(url)
        if js_indicator:
            WebDriverWait(driver, timeout).until(
                EC.presence_of_element_located((By.CSS_SELECTOR, js_indicator))
            )
        else:
            time.sleep(3)
        soup = BeautifulSoup(driver.page_source, 'html.parser')
        print(f'  [selenium] {url}')
        return soup
    finally:
        driver.quit()


# ── Test smart_fetch on both types of pages ──
print('Testing smart_fetch:')
print()

# Static page — requests works fine
soup1 = smart_fetch('https://quotes.toscrape.com/', js_indicator='div.quote')
print(f'  Quotes found: {len(soup1.find_all("div", class_="quote"))}')
print()

# JS page — requests will miss content, Selenium takes over
soup2 = smart_fetch('https://quotes.toscrape.com/js/', js_indicator='div.quote')
print(f'  JS Quotes found: {len(soup2.find_all("div", class_="quote"))}')

## 🔗 `requests.Session()` — The Right Way for Multi-Request Scrapers

Every time you call `requests.get()`, Python opens a **new TCP connection**.
With 50+ pages, that's 50+ connection handshakes — slow and wasteful.

`requests.Session()` reuses the same connection for all requests:

```
Without Session:  request → open conn → get → close → repeat 50×
With Session:     open conn once → get → get → get → close (50 requests, 1 conn)
```

```python
session = requests.Session()
session.headers.update(HEADERS)   # Set once, applies to ALL requests

r1 = session.get(url1)            # Reuses connection
r2 = session.get(url2)            # Still reusing
r3 = session.get(url3)            # Still reusing

session.close()   # Close when done
```

**Speed improvement**: 20–30% faster on multi-page scrapers.
**Bonus**: Session also persists cookies automatically — perfect for login scenarios.

In [ ]:
# SESSION DEMO — fetch_with_session() replaces fetch_with_retry()

import time

def make_session():
    """Create a configured requests Session to reuse across all pages."""
    session = requests.Session()
    session.headers.update(HEADERS)
    return session


def fetch_with_session(session, url, max_retries=3):
    """
    Fetch a URL using a shared Session.
    Pass the SAME session object when scraping multiple pages.
    """
    for attempt in range(1, max_retries + 1):
        try:
            r = session.get(url, timeout=10)
            r.raise_for_status()
            logger.info(f'[Session] Fetched (attempt {attempt}): {url}')
            return BeautifulSoup(r.text, 'html.parser')
        except requests.exceptions.HTTPError as e:
            logger.error(f'HTTP {e.response.status_code}: {url}')
            if e.response.status_code == 404: return None
        except requests.exceptions.RequestException as e:
            logger.warning(f'Attempt {attempt} failed: {e}')
            if attempt < max_retries:
                time.sleep(2 ** (attempt - 1))
    return None


# ── Demo: Scrape 3 pages using the SAME Session ──
session = make_session()

urls = [
    'https://quotes.toscrape.com/page/1/',
    'https://quotes.toscrape.com/page/2/',
    'https://quotes.toscrape.com/page/3/',
]

print('Scraping 3 pages with shared Session:')
for url in urls:
    soup = fetch_with_session(session, url)
    if soup:
        count = len(soup.find_all('div', class_='quote'))
        print(f'  {url[-8:]:12s} -> {count} quotes')
    time.sleep(1)

session.close()   # Always close when done!
print('Session closed.')


## 🧹 Cell 3 — Reusable Data Cleaner

In real projects, you extract data from 10+ fields per item. Writing cleaning code inline every time is messy. Build **one cleaner module** used everywhere.

### What a Cleaner Does

```
Raw scraped text      ->   Cleaned, typed, validated value
─────────────────────────────────────────────────────────
'  Rs 1,299.00  '    ->   1299.0  (float)
'⭐⭐⭐ (4.5/5)'      ->   4.5     (float)
'  Python Basics  '  ->   'Python Basics'  (stripped string)
None                 ->   'N/A'   (default, no crash)
'In stock'           ->   True    (boolean)
```

### Design Principle
Each cleaner function:
- Takes a raw tag or string
- Returns a clean Python value (str, float, bool)
- **Never crashes** — returns a default instead

In [ ]:
# CELL 3 — Reusable Data Cleaner Module

import re

def safe_text(tag, default='N/A'):
    """Extract and strip text from a BS4 tag. Returns default if tag is None."""
    return tag.get_text(strip=True) if tag else default


def safe_attr(tag, attr, default=''):
    """Safely get an attribute from a BS4 tag. Returns default if tag is None."""
    return tag.get(attr, default) if tag else default


def to_float(text, default=0.0):
    """Convert a scraped price string to float. Handles £, $, Rs, commas."""
    if not text:
        return default
    try:
        cleaned = re.sub(r'[^\d.]', '', text)   # Remove everything except digits and '.'
        return float(cleaned) if cleaned else default
    except (ValueError, TypeError):
        return default


def to_bool_stock(text):
    """Convert 'In stock' / 'Out of stock' text to True/False."""
    return 'in stock' in text.lower() if text else False


def clean_url(href, base_url=''):
    """Build absolute URL from a relative href."""
    from urllib.parse import urljoin
    return urljoin(base_url, href) if href else ''


def word_count_rating(text):
    """Convert word rating to number. 'Three' -> 3."""
    mapping = {'one': 1, 'two': 2, 'three': 3, 'four': 4, 'five': 5}
    return mapping.get(text.lower(), 0) if text else 0


# ── Test all cleaners ──
print('Cleaner tests:')
print('-' * 40)
print(f'safe_text(None)        : {safe_text(None)}')
print(f'to_float("£51.77")     : {to_float("£51.77")}')
print(f'to_float("Rs 1,299.00"): {to_float("Rs 1,299.00")}')
print(f'to_float(None)         : {to_float(None)}')
print(f'to_bool_stock("In stock")     : {to_bool_stock("In stock")}')
print(f'to_bool_stock("Out of stock") : {to_bool_stock("Out of stock")}')
print(f'word_count_rating("Three")    : {word_count_rating("Three")}')
print(f'clean_url("../cat/book.html", "https://books.toscrape.com/catalogue/"):')
print(f'  {clean_url("../cat/book.html", "https://books.toscrape.com/catalogue/")}')

## ✅ Data Validation — Catch Bad Data Before It Enters Your DB

Raw scraped data can have issues:
- Price came as `£` only (no number) → `to_float()` returns `0.0`
- Title was truncated → stored as `'A Light in...'` not full title
- Field missing on some pages → stored as `'N/A'`

A **validation step** catches these before saving.

```python
def validate_book(book):
    errors = []
    if not book.get('title') or book['title'] == 'N/A':
        errors.append('Missing title')
    if book.get('price', 0) <= 0:
        errors.append(f'Invalid price: {book.get("price")}')
    if book.get('rating', 0) not in range(0, 6):
        errors.append(f'Invalid rating: {book.get("rating")}')
    return errors   # Empty list = valid
```

### Where to Validate
```
fetch() -> parse() -> validate() -> save()   ← validate BEFORE save
                           ↓
                    log invalid items
                    skip or fix them
```

In [ ]:
# DATA VALIDATION

def validate_item(item, rules):
    """
    Validate a scraped item against a rules dict.
    rules = {'field_name': validation_function}
    Returns list of error strings. Empty = valid.
    """
    errors = []
    for field, check_fn in rules.items():
        val = item.get(field)
        try:
            if not check_fn(val):
                errors.append(f'{field}={repr(val)} failed validation')
        except Exception as e:
            errors.append(f'{field} error: {e}')
    return errors


# Define validation rules for quotes
QUOTE_RULES = {
    'quote' : lambda v: v and v != 'N/A' and len(v) > 5,
    'author': lambda v: v and v != 'N/A',
    'tags'  : lambda v: isinstance(v, (list, str)),
}

# Test with real data
test_items = [
    {'quote': '“The world is a book”', 'author': 'Augustine', 'tags': ['travel']},
    {'quote': 'N/A',                       'author': 'Unknown',   'tags': []},
    {'quote': '“Be yourself”',               'author': 'N/A',       'tags': ['life']},
    {'quote': '',                           'author': '',          'tags': []},
]

print('Validation results:')
print('-' * 55)
valid_count   = 0
invalid_count = 0

for item in test_items:
    errors = validate_item(item, QUOTE_RULES)
    if errors:
        print(f'  ❌ INVALID: {item["quote"][:30]:30s} | {errors}')
        invalid_count += 1
    else:
        print(f'  ✅ VALID  : {item["quote"][:30]:30s}')
        valid_count += 1

print()
print(f'Valid: {valid_count} | Invalid: {invalid_count}')
print()

# ── Context manager for DB (better pattern) ──
print('Context manager DB pattern:')
with sqlite3.connect('quotes_pipeline.db') as conn:   # auto-closes!
    count = conn.execute('SELECT COUNT(*) FROM quotes').fetchone()[0]
    print(f'  DB has {count} quotes (connection auto-closed)')


## 🔁 Cell 4 — Retry with Exponential Backoff

Networks are unreliable. A single failed request shouldn't kill your entire scraper.
**Retry with backoff** means: wait a little longer after each failure before trying again.

```
Attempt 1 -> fails -> wait 1s
Attempt 2 -> fails -> wait 2s
Attempt 3 -> fails -> wait 4s
Attempt 4 -> fails -> give up, return None
```

This is called **exponential backoff** — each wait doubles. It's the industry standard.

```python
def fetch_with_retry(url, max_retries=3):
    for attempt in range(max_retries):
        try:
            r = requests.get(url, timeout=10)
            r.raise_for_status()
            return r
        except requests.exceptions.RequestException:
            wait = 2 ** attempt      # 1s, 2s, 4s
            time.sleep(wait)
    return None                      # All retries failed
```

In [ ]:
# CELL 4 — Retry with Exponential Backoff

def fetch_with_retry(url, headers=None, max_retries=3, timeout=10):
    """
    Fetch a URL with automatic retry and exponential backoff.
    Returns BeautifulSoup on success, None after all retries fail.
    """
    headers = headers or HEADERS

    for attempt in range(1, max_retries + 1):
        try:
            r = requests.get(url, headers=headers, timeout=timeout)
            r.raise_for_status()
            return BeautifulSoup(r.text, 'html.parser')

        except requests.exceptions.HTTPError as e:
            print(f'  Attempt {attempt}/{max_retries} | HTTP Error: {e}')
            # Don't retry 404 - page genuinely doesn't exist
            if e.response.status_code == 404:
                print('  404 - Not retrying.')
                return None

        except requests.exceptions.RequestException as e:
            print(f'  Attempt {attempt}/{max_retries} | Error: {e}')

        if attempt < max_retries:
            wait = 2 ** (attempt - 1)    # 1s, 2s, 4s
            print(f'  Retrying in {wait}s...')
            time.sleep(wait)

    print(f'  All {max_retries} retries failed for: {url}')
    return None


# ── Test 1: Valid URL ──
print('Test 1: Valid URL')
soup = fetch_with_retry('https://quotes.toscrape.com')
if soup:
    count = len(soup.find_all('div', class_='quote'))
    print(f'  Success! Found {count} quotes')
print()

# ── Test 2: Non-existent page (404 - no retry) ──
print('Test 2: 404 URL (should not retry)')
soup = fetch_with_retry('https://quotes.toscrape.com/this-page-does-not-exist')
print(f'  Result: {soup}')   # None
print()

# ── Test 3: Bad domain (will retry) ──
print('Test 3: Bad domain (will retry 3 times)')
soup = fetch_with_retry('https://this-domain-does-not-exist-xyz.com', max_retries=2)
print(f'  Result: {soup}')   # None after retries

## 📝 Cell 5 — Logging — Track Everything Your Scraper Does

Using `print()` for debugging is fine locally. In production, use Python's `logging` module:

| Feature | `print()` | `logging` |
|---------|-----------|----------|
| Log levels | ❌ No levels | ✅ DEBUG, INFO, WARNING, ERROR |
| Save to file | ❌ Only terminal | ✅ File + terminal simultaneously |
| Timestamps | ❌ Manual | ✅ Automatic |
| Turn off easily | ❌ Delete each print | ✅ Change one log level |

### Log Levels (in order of severity)
```python
logging.DEBUG    # Fine-grained detail (for development)
logging.INFO     # General progress   (page scraped, item saved)
logging.WARNING  # Something unusual  (missing field, retry)
logging.ERROR    # Something failed   (request failed, parse error)
```

### Setup Pattern
```python
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s | %(levelname)s | %(message)s',
    handlers=[
        logging.FileHandler('scraper.log'),   # Save to file
        logging.StreamHandler()               # Also show in terminal
    ]
)
logger = logging.getLogger(__name__)
```

In [ ]:
# CELL 5 — Logging Setup

import logging

# Setup logger — runs once at the top of any scraper script
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s | %(levelname)-8s | %(message)s',
    datefmt='%H:%M:%S',
    handlers=[
        logging.FileHandler('scraper.log', mode='w'),   # Save to file
        logging.StreamHandler()                          # Show in terminal
    ]
)
logger = logging.getLogger('scraper')


# ── Updated fetch_with_retry using logger ──
def fetch_logged(url, max_retries=3):
    """fetch_with_retry but with proper logging."""
    for attempt in range(1, max_retries + 1):
        try:
            r = requests.get(url, headers=HEADERS, timeout=10)
            r.raise_for_status()
            logger.info(f'Fetched [{r.status_code}]: {url}')
            return BeautifulSoup(r.text, 'html.parser')

        except requests.exceptions.HTTPError as e:
            logger.error(f'HTTP {e.response.status_code} on attempt {attempt}: {url}')
            if e.response.status_code == 404:
                return None

        except requests.exceptions.RequestException as e:
            logger.warning(f'Attempt {attempt} failed: {e}')

        if attempt < max_retries:
            wait = 2 ** (attempt - 1)
            logger.warning(f'Retrying in {wait}s...')
            time.sleep(wait)

    logger.error(f'All retries exhausted for: {url}')
    return None


# ── Test logging ──
logger.info('Scraper started')
logger.debug('Debug level - not shown at INFO level')

soup = fetch_logged('https://quotes.toscrape.com')
if soup:
    count = len(soup.find_all('div', class_='quote'))
    logger.info(f'Parsed {count} quotes from page')
else:
    logger.error('Failed to fetch page')

fetch_logged('https://quotes.toscrape.com/nonexistent')  # 404 test

logger.info('Done. Check scraper.log for full logs.')
print()
print('Log file content:')
with open('scraper.log') as f:
    print(f.read())

## 🗄️ Cell 6 — SQLite — Real Database Storage

CSV and JSON are fine for small datasets. For anything larger or queried repeatedly, use **SQLite** — a real database that lives in one `.db` file, no server needed.

### Why SQLite over CSV?

| | CSV | SQLite |
|-|-----|--------|
| Filter by price < 500 | Read all, filter in Python | `SELECT * WHERE price < 500` |
| Avoid duplicates | Manual check | `INSERT OR IGNORE` |
| Update one record | Rewrite entire file | `UPDATE SET price=... WHERE id=...` |
| Query 100k rows | Slow | Fast (indexed) |

### Basic Pattern
```python
import sqlite3

conn = sqlite3.connect('data.db')   # Creates file if not exists
c = conn.cursor()

# Create table
c.execute('''
    CREATE TABLE IF NOT EXISTS books (
        id     INTEGER PRIMARY KEY AUTOINCREMENT,
        title  TEXT UNIQUE,        <- UNIQUE prevents duplicates!
        price  REAL,
        rating INTEGER
    )
''')

# Insert (ignore if duplicate title)
c.execute('INSERT OR IGNORE INTO books (title, price, rating) VALUES (?,?,?)',
          ('Python Book', 499.0, 5))

conn.commit()
conn.close()
```

In [ ]:
# CELL 6 — SQLite Storage

import sqlite3

DB_FILE = 'scraper_data.db'

def init_db():
    """Create database and tables if they don't exist."""
    conn = sqlite3.connect(DB_FILE)
    c = conn.cursor()
    c.execute('''
        CREATE TABLE IF NOT EXISTS books (
            id           INTEGER PRIMARY KEY AUTOINCREMENT,
            title        TEXT UNIQUE,
            price        REAL,
            rating       INTEGER,
            availability TEXT,
            scraped_at   DATETIME DEFAULT CURRENT_TIMESTAMP
        )
    ''')
    conn.commit()
    conn.close()
    logger.info(f'Database initialised: {DB_FILE}')


def save_books(books_list):
    """Save a list of book dicts to SQLite. Skips duplicates automatically."""
    conn = sqlite3.connect(DB_FILE)
    c = conn.cursor()
    saved = 0
    for book in books_list:
        try:
            c.execute(
                'INSERT OR IGNORE INTO books (title, price, rating, availability) VALUES (?,?,?,?)',
                (book.get('title'), book.get('price'), book.get('rating'), book.get('availability'))
            )
            if c.rowcount > 0:   # rowcount = 0 means it was a duplicate (ignored)
                saved += 1
        except Exception as e:
            logger.error(f'DB insert failed for "{book.get("title")}": {e}')
    conn.commit()
    conn.close()
    return saved


def query_books(min_price=None, max_price=None, min_rating=None):
    """Query books from DB with optional filters."""
    conn = sqlite3.connect(DB_FILE)
    conn.row_factory = sqlite3.Row   # Returns dict-like rows
    c = conn.cursor()
    query = 'SELECT * FROM books WHERE 1=1'
    params = []
    if min_price  is not None: query += ' AND price >= ?';  params.append(min_price)
    if max_price  is not None: query += ' AND price <= ?';  params.append(max_price)
    if min_rating is not None: query += ' AND rating >= ?'; params.append(min_rating)
    c.execute(query + ' ORDER BY price', params)
    rows = [dict(r) for r in c.fetchall()]
    conn.close()
    return rows


# ── Test: Create DB, insert books, query them ──
init_db()

sample_books = [
    {'title': 'Python Crash Course',   'price': 499.0, 'rating': 5, 'availability': 'In stock'},
    {'title': 'Automate Boring Stuff', 'price': 399.0, 'rating': 4, 'availability': 'In stock'},
    {'title': 'Fluent Python',          'price': 799.0, 'rating': 5, 'availability': 'In stock'},
    {'title': 'Python Crash Course',   'price': 499.0, 'rating': 5, 'availability': 'In stock'},  # Duplicate!
]

saved = save_books(sample_books)
print(f'Saved {saved} new books (1 duplicate skipped)')
print()

# Query all books
all_books = query_books()
print(f'Total books in DB: {len(all_books)}')
for b in all_books:
    print(f'  [{b["id"]}] {b["title"]:25s} | £{b["price"]:6.2f} | Rating: {b["rating"]} ⭐')
print()

# Query with filter
cheap = query_books(max_price=500, min_rating=4)
print(f'Cheap (<=500) and good (>=4★): {len(cheap)} books')
for b in cheap:
    print(f'  {b["title"]} @ £{b["price"]}')

## 🔍 Cell 7 — Deduplication — Never Scrape the Same Item Twice

When scraping many pages or re-running a scraper, you'll hit the same items repeatedly.
Deduplication ensures you only process and store **new** data.

### Two Levels of Deduplication

**Level 1 — In-memory (same run)**
```python
seen_urls = set()         # Set is O(1) lookup — very fast

if url in seen_urls:
    continue              # Already processed this URL
seen_urls.add(url)
```

**Level 2 — Database (across runs)**
```python
# SQLite's INSERT OR IGNORE handles this automatically
# if title is UNIQUE in the schema:
c.execute('INSERT OR IGNORE INTO books (title, price) VALUES (?,?)', (title, price))
# Duplicate title? Silently skipped. No error. No duplicate.
```

### Fingerprinting
For items without a unique field, create a hash fingerprint:
```python
import hashlib
fingerprint = hashlib.md5(f'{title}{price}'.encode()).hexdigest()
```

In [ ]:
# CELL 7 — Deduplication in Action

import hashlib

def make_fingerprint(*args):
    """Create a unique hash from one or more strings."""
    combined = '|'.join(str(a).lower().strip() for a in args)
    return hashlib.md5(combined.encode()).hexdigest()


# ── Simulate scraping 3 pages where page 3 has duplicates ──
raw_data_from_pages = [
    # Page 1
    [{'title': 'Python Crash Course', 'price': 499.0},
     {'title': 'Automate Boring Stuff', 'price': 399.0}],
    # Page 2
    [{'title': 'Fluent Python', 'price': 799.0},
     {'title': 'Learning Python', 'price': 599.0}],
    # Page 3 — has two items we've seen before!
    [{'title': 'Python Crash Course', 'price': 499.0},    # DUPLICATE
     {'title': 'Data Science Handbook', 'price': 649.0}],
]

seen = set()    # In-memory seen set
unique_items = []

for page_num, page_items in enumerate(raw_data_from_pages, 1):
    print(f'Processing page {page_num}:')
    for item in page_items:
        fp = make_fingerprint(item['title'], item['price'])
        if fp in seen:
            print(f'  ⚠️  DUPLICATE skipped: "{item["title"]}"')
            continue
        seen.add(fp)
        unique_items.append(item)
        print(f'  ✅  New item added  : "{item["title"]}"')

print()
print(f'Total raw items   : {sum(len(p) for p in raw_data_from_pages)}')
print(f'Unique items kept : {len(unique_items)}')
print(f'Duplicates skipped: {sum(len(p) for p in raw_data_from_pages) - len(unique_items)}')

## 💾 Checkpoint & Resume — Never Start Over After a Crash

Long scrapers (50+ pages, hours of runtime) **will** crash eventually:
- Network drops
- Power outage
- Site temporarily down

Without a checkpoint, you lose all progress and restart from page 1.
With a checkpoint, you resume from where you stopped.

```python
CHECKPOINT_FILE = 'scraper_checkpoint.json'

# Save checkpoint after each page:
checkpoint = {'last_page': 35, 'next_url': 'https://...page-36.html', 'total_saved': 700}
json.dump(checkpoint, open(CHECKPOINT_FILE, 'w'))

# On restart, load and continue:
if os.path.exists(CHECKPOINT_FILE):
    checkpoint = json.load(open(CHECKPOINT_FILE))
    start_url  = checkpoint['next_url']     # Resume from page 36
    page_num   = checkpoint['last_page'] + 1
```

In [ ]:
# CHECKPOINT & RESUME PATTERN

import os

CHECKPOINT_FILE = 'scraper_checkpoint.json'


def save_checkpoint(page_num, next_url, total_saved):
    """Save current progress to file."""
    data = {
        'last_page'  : page_num,
        'next_url'   : next_url,
        'total_saved': total_saved,
        'saved_at'   : datetime.now().strftime('%Y-%m-%d %H:%M:%S')
    }
    with open(CHECKPOINT_FILE, 'w') as f:
        json.dump(data, f, indent=2)


def load_checkpoint():
    """Load checkpoint if exists. Returns None if fresh start."""
    if os.path.exists(CHECKPOINT_FILE):
        with open(CHECKPOINT_FILE) as f:
            cp = json.load(f)
        logger.info(f'Resuming from checkpoint: page {cp["last_page"]+1}, {cp["total_saved"]} already saved')
        return cp
    logger.info('No checkpoint found. Starting fresh.')
    return None


def clear_checkpoint():
    """Delete checkpoint after successful completion."""
    if os.path.exists(CHECKPOINT_FILE):
        os.remove(CHECKPOINT_FILE)
        logger.info('Checkpoint cleared (scrape complete).')


# ── Demo ──
from datetime import datetime

# Simulate saving checkpoint at page 5
save_checkpoint(
    page_num    = 5,
    next_url    = 'https://quotes.toscrape.com/page/6/',
    total_saved = 50
)
print('Checkpoint saved!')

# Load it back
cp = load_checkpoint()
if cp:
    print(f'\nCheckpoint loaded:')
    for k, v in cp.items():
        print(f'  {k:15s}: {v}')
    print(f'\nScraper would resume from: page {cp["last_page"]+1}')
    print(f'Starting URL             : {cp["next_url"]}')

# Clean up
clear_checkpoint()
print('\nCheckpoint cleared after demo.')


## 🏭 Cell 8 — Putting It All Together: The Full Pipeline

All the pieces built so far, wired into one complete scraping pipeline:

```
                    ScrapingPipeline
                          │
              ┌───────────┼───────────┐
              │           │           │
         smart_fetch  clean_data  save_to_db
              │           │           │
       requests or     type conv   SQLite with
       Selenium        + None safe  deduplication
              │
        retry + backoff
              │
           logging
```

### Usage Pattern
```python
pipeline = ScrapingPipeline(db_file='data.db')

for page in range(1, 6):
    url  = f'https://site.com/page/{page}'
    data = pipeline.scrape_page(url)
    pipeline.save(data)
    time.sleep(random.uniform(1, 2))
```

In [ ]:
# CELL 8 — Full Scraping Pipeline Class

class ScrapingPipeline:
    """
    A complete, production-ready scraping pipeline.
    Combines: smart fetch + retry + logging + cleaning + dedup + SQLite.
    """

    def __init__(self, db_file='pipeline.db', max_retries=3):
        self.db_file     = db_file
        self.max_retries = max_retries
        self.seen        = set()    # In-memory dedup
        self._init_db()
        logger.info(f'Pipeline initialized. DB: {db_file}')

    def _init_db(self):
        conn = sqlite3.connect(self.db_file)
        conn.execute('''
            CREATE TABLE IF NOT EXISTS quotes (
                id         INTEGER PRIMARY KEY AUTOINCREMENT,
                quote      TEXT UNIQUE,
                author     TEXT,
                tags       TEXT,
                page       INTEGER,
                scraped_at DATETIME DEFAULT CURRENT_TIMESTAMP
            )
        ''')
        conn.commit()
        conn.close()

    def fetch(self, url):
        """Fetch with retry + logging."""
        for attempt in range(1, self.max_retries + 1):
            try:
                r = requests.get(url, headers=HEADERS, timeout=10)
                r.raise_for_status()
                logger.info(f'Fetched (attempt {attempt}): {url}')
                return BeautifulSoup(r.text, 'html.parser')
            except requests.exceptions.HTTPError as e:
                logger.error(f'HTTP {e.response.status_code}: {url}')
                if e.response.status_code == 404: return None
            except requests.exceptions.RequestException as e:
                logger.warning(f'Attempt {attempt} failed: {e}')
                if attempt < self.max_retries:
                    time.sleep(2 ** (attempt - 1))
        return None

    def parse_quotes(self, soup, page_num=1):
        """Extract + clean quotes from a BS4 soup object."""
        items = []
        for box in soup.find_all('div', class_='quote'):
            text   = safe_text(box.find('span', class_='text'))
            author = safe_text(box.find('small', class_='author'))
            tags   = [safe_text(t) for t in box.find_all('a', class_='tag')]

            fp = make_fingerprint(text, author)
            if fp in self.seen:
                logger.debug(f'Duplicate skipped: {author}')
                continue
            self.seen.add(fp)
            items.append({'quote': text, 'author': author,
                          'tags': json.dumps(tags), 'page': page_num})
        return items

    def save(self, items):
        """Save items to SQLite, skip duplicates."""
        conn = sqlite3.connect(self.db_file)
        saved = 0
        for item in items:
            c = conn.execute(
                'INSERT OR IGNORE INTO quotes (quote, author, tags, page) VALUES (?,?,?,?)',
                (item['quote'], item['author'], item['tags'], item['page'])
            )
            if c.rowcount > 0: saved += 1
        conn.commit()
        conn.close()
        logger.info(f'Saved {saved}/{len(items)} items')
        return saved


# ── Run the pipeline ──
pipeline = ScrapingPipeline(db_file='quotes_pipeline.db')

for page in range(1, 4):    # Scrape first 3 pages
    url  = f'https://quotes.toscrape.com/page/{page}/'
    soup = pipeline.fetch(url)
    if soup:
        items = pipeline.parse_quotes(soup, page_num=page)
        pipeline.save(items)
    time.sleep(random.uniform(1, 2))

# Verify DB
conn = sqlite3.connect('quotes_pipeline.db')
count = conn.execute('SELECT COUNT(*) FROM quotes').fetchone()[0]
top   = conn.execute('SELECT author, COUNT(*) c FROM quotes GROUP BY author ORDER BY c DESC LIMIT 3').fetchall()
conn.close()

print(f'\nTotal quotes in DB : {count}')
print('Top 3 authors:')
for author, cnt in top:
    print(f'  {author}: {cnt} quotes')

---
# 🧪 Real Projects — Production Pipeline in Action

---

## 💼 Project 6 — Job Listings Scraper

**Website**: `https://realpython.github.io/fake-jobs/`
**Goal**: Scrape all job listings (title, company, location, date) → save to SQLite + CSV.

This site is purpose-built for scraping practice — perfect for learning.

### Step 1 — Inspect the Page First!

Open the URL → Right-click a job card → Inspect:

```html
<div class="card-content">
  <h2 class="title is-5">Senior Python Developer</h2>
  <h3 class="subtitle is-6 company">Payne, Roberts and Davis</h3>
  <p class="location">Stewartbury, AA</p>
  <p class="is-small has-text-grey">
    <time datetime="2021-04-08">2021-04-08</time>
  </p>
</div>
<div class="card-footer">
  <a href="#">Apply</a>
</div>
```

### Strategy
```
1. fetch_with_retry()      → get the page
2. find_all card-content   → all job cards
3. safe_text() per field   → no crashes
4. Deduplication           → fingerprint by title+company
5. SQLite + CSV            → dual storage
6. Logger throughout       → track everything
```

In [ ]:
# PROJECT 6 — Job Listings Scraper (Full Pipeline)

import csv
import sqlite3
import hashlib

# ── DB Setup ──
JOB_DB = 'jobs.db'

def init_jobs_db():
    conn = sqlite3.connect(JOB_DB)
    conn.execute('''
        CREATE TABLE IF NOT EXISTS jobs (
            id          INTEGER PRIMARY KEY AUTOINCREMENT,
            title       TEXT,
            company     TEXT,
            location    TEXT,
            posted_date TEXT,
            fingerprint TEXT UNIQUE,
            scraped_at  DATETIME DEFAULT CURRENT_TIMESTAMP
        )
    ''')
    conn.commit()
    conn.close()


def scrape_jobs(url):
    """Fetch and parse all job listings from the page."""
    logger.info(f'Scraping jobs from: {url}')

    soup = fetch_with_retry(url)
    if not soup:
        logger.error('Failed to fetch jobs page')
        return []

    cards = soup.find_all('div', class_='card-content')
    logger.info(f'Found {len(cards)} job cards')

    jobs = []
    for card in cards:
        title    = safe_text(card.find('h2', class_='title'))
        company  = safe_text(card.find('h3', class_='company'))
        location = safe_text(card.find('p',  class_='location'))
        date_tag = card.find('time')
        date     = date_tag.get('datetime', safe_text(date_tag)) if date_tag else 'N/A'

        fp = hashlib.md5(f'{title}|{company}'.encode()).hexdigest()

        jobs.append({
            'title'      : title,
            'company'    : company,
            'location'   : location,
            'posted_date': date,
            'fingerprint': fp
        })

    return jobs


def save_jobs_db(jobs):
    """Save jobs to SQLite, skip duplicates via fingerprint."""
    conn = sqlite3.connect(JOB_DB)
    saved = 0
    for job in jobs:
        c = conn.execute(
            'INSERT OR IGNORE INTO jobs (title,company,location,posted_date,fingerprint)'
            ' VALUES (?,?,?,?,?)',
            (job['title'], job['company'], job['location'],
             job['posted_date'], job['fingerprint'])
        )
        if c.rowcount > 0: saved += 1
    conn.commit()
    conn.close()
    return saved


def save_jobs_csv(jobs, filepath='jobs.csv'):
    """Save jobs to CSV file."""
    if not jobs: return
    with open(filepath, 'w', newline='', encoding='utf-8') as f:
        writer = csv.DictWriter(f, fieldnames=['title','company','location','posted_date'])
        writer.writeheader()
        writer.writerows([{k: v for k,v in j.items() if k != 'fingerprint'} for j in jobs])
    logger.info(f'CSV saved: {filepath}')


# ── Run the full pipeline ──
init_jobs_db()

url  = 'https://realpython.github.io/fake-jobs/'
jobs = scrape_jobs(url)

# Save to both SQLite and CSV
new_count = save_jobs_db(jobs)
save_jobs_csv(jobs)

logger.info(f'Pipeline complete: {new_count} new jobs saved out of {len(jobs)} found')

# ── Print summary ──
print(f'\nResults:')
print(f'  Total jobs found : {len(jobs)}')
print(f'  New jobs saved   : {new_count}')
print()
print('Sample (first 5):')
print(f'{"Title":<35} {"Company":<30} {"Location"}')
print('-' * 80)
for job in jobs[:5]:
    print(f'{job["title"][:34]:<35} {job["company"][:29]:<30} {job["location"]}')

# ── Quick DB query ──
conn = sqlite3.connect(JOB_DB)
top_locations = conn.execute(
    'SELECT location, COUNT(*) c FROM jobs GROUP BY location ORDER BY c DESC LIMIT 5'
).fetchall()
conn.close()

print()
print('Top 5 locations:')
for loc, count in top_locations:
    print(f'  {loc}: {count} jobs')

## 📈 Project 7 — Price Tracker with Change Detection

**Website**: `https://books.toscrape.com`
**Goal**: Scrape book prices, store them with timestamps, detect price changes.

This is one of the most common real-world use cases for web scraping:
- E-commerce price monitoring
- Flight price tracking
- Real estate price changes

### How Price Tracking Works

```
Run 1 (today):
  Python Crash Course -> £49.99  <- stored with timestamp

Run 2 (tomorrow):
  Python Crash Course -> £39.99  <- NEW price stored
  Compare with last price -> PRICE DROP of £10.00! 🔔
```

### DB Schema for Price Tracking

```sql
-- One row per price observation (not unique by title!)
CREATE TABLE price_history (
    id         INTEGER PRIMARY KEY,
    title      TEXT,
    price      REAL,
    scraped_at DATETIME DEFAULT CURRENT_TIMESTAMP
)
```

**Key difference from other tables**: we keep ALL historical rows, not just the latest.
This lets us query: `SELECT * WHERE title='...' ORDER BY scraped_at` to see price history.

In [ ]:
# PROJECT 7 — Price Tracker with Change Detection

PRICE_DB = 'price_tracker.db'

def init_price_db():
    conn = sqlite3.connect(PRICE_DB)
    conn.execute('''
        CREATE TABLE IF NOT EXISTS price_history (
            id         INTEGER PRIMARY KEY AUTOINCREMENT,
            title      TEXT,
            price      REAL,
            scraped_at DATETIME DEFAULT CURRENT_TIMESTAMP
        )
    ''')
    conn.commit()
    conn.close()


def scrape_prices(url):
    """Scrape book titles and prices from books.toscrape.com."""
    soup = fetch_with_retry(url)
    if not soup:
        logger.error('Failed to fetch prices page')
        return []

    books = []
    for card in soup.find_all('article', class_='product_pod'):
        h3        = card.find('h3')
        title     = h3.find('a')['title'] if h3 and h3.find('a') else 'N/A'
        price_tag = card.find('p', class_='price_color')
        price     = to_float(safe_text(price_tag))
        books.append({'title': title, 'price': price})

    logger.info(f'Scraped {len(books)} book prices')
    return books


def save_prices(books):
    """Save ALL prices (every run creates new rows — this is intentional!)."""
    conn = sqlite3.connect(PRICE_DB)
    conn.executemany(
        'INSERT INTO price_history (title, price) VALUES (?,?)',
        [(b['title'], b['price']) for b in books]
    )
    conn.commit()
    conn.close()


def detect_changes():
    """
    Compare latest prices to previous prices.
    Returns list of books whose price changed.
    """
    conn = sqlite3.connect(PRICE_DB)

    # Get the 2 most recent distinct scrape times
    times = conn.execute(
        "SELECT DISTINCT DATE(scraped_at) FROM price_history ORDER BY scraped_at DESC LIMIT 2"
    ).fetchall()

    if len(times) < 2:
        conn.close()
        return []   # Need at least 2 runs to detect changes

    latest, previous = times[0][0], times[1][0]

    # Get prices for both runs
    latest_prices   = {r[0]: r[1] for r in conn.execute(
        "SELECT title, price FROM price_history WHERE DATE(scraped_at)=?", (latest,)
    ).fetchall()}
    previous_prices = {r[0]: r[1] for r in conn.execute(
        "SELECT title, price FROM price_history WHERE DATE(scraped_at)=?", (previous,)
    ).fetchall()}
    conn.close()

    # Find changes
    changes = []
    for title, new_price in latest_prices.items():
        if title in previous_prices:
            old_price = previous_prices[title]
            if abs(new_price - old_price) > 0.01:   # Price changed
                diff = new_price - old_price
                changes.append({
                    'title'    : title,
                    'old_price': old_price,
                    'new_price': new_price,
                    'change'   : diff,
                    'direction': '📉 DROP' if diff < 0 else '📈 RISE'
                })
    return sorted(changes, key=lambda x: abs(x['change']), reverse=True)


# ── Run the Price Tracker ──
init_price_db()

url   = 'https://books.toscrape.com'
books = scrape_prices(url)
save_prices(books)

print(f'Scraped and saved {len(books)} book prices')
print()

# Show current prices (top 10 cheapest)
sorted_books = sorted(books, key=lambda x: x['price'])
print('Top 5 cheapest books right now:')
print(f'{"Title":<45} {"Price":>8}')
print('-' * 55)
for b in sorted_books[:5]:
    print(f'{b["title"][:44]:<45} £{b["price"]:>6.2f}')
print()

# ── Simulate a price change for demo ──
# (In real life, you'd run this script again tomorrow)
print('Simulating a second run with modified prices...')
modified = []
for i, b in enumerate(books):
    # Artificially change price of every 5th book for demo
    new_price = b['price'] - 5.0 if i % 5 == 0 else b['price']
    modified.append({'title': b['title'], 'price': new_price})
save_prices(modified)
print(f'Second run saved ({len(modified)} prices)')
print()

# Detect changes
changes = detect_changes()
if changes:
    print(f'🔔 Price changes detected: {len(changes)}')
    print(f'{"Title":<40} {"Old":>8} {"New":>8} {"Change":>8} {""}')
    print('-' * 75)
    for c in changes[:8]:
        print(f'{c["title"][:39]:<40} £{c["old_price"]:>6.2f} £{c["new_price"]:>6.2f}'
              f' {c["change"]:>+7.2f} {c["direction"]}')
else:
    print('No price changes detected (need 2 different scrape dates)')

## ⚡ Production Patterns — Cheat Sheet

### Smart Fetcher
```python
def smart_fetch(url, js_indicator=None):
    """Try requests first, fall back to Selenium if needed."""
    r = requests.get(url, headers=HEADERS, timeout=10)
    soup = BeautifulSoup(r.text, 'html.parser')
    if js_indicator and not soup.select_one(js_indicator):
        # JS page — use Selenium
        driver = get_driver(headless=True)
        driver.get(url)
        soup = BeautifulSoup(driver.page_source, 'html.parser')
        driver.quit()
    return soup
```

### Retry with Backoff
```python
for attempt in range(1, max_retries + 1):
    try:
        r = requests.get(url, timeout=10)
        r.raise_for_status()
        return BeautifulSoup(r.text, 'html.parser')
    except requests.exceptions.RequestException:
        time.sleep(2 ** (attempt - 1))   # 1s, 2s, 4s
return None
```

### Safe Extraction
```python
def safe_text(tag, default='N/A'):
    return tag.get_text(strip=True) if tag else default

def to_float(text, default=0.0):
    cleaned = re.sub(r'[^\d.]', '', text or '')
    return float(cleaned) if cleaned else default
```

### Deduplication
```python
seen = set()
fp = hashlib.md5(f'{title}|{company}'.encode()).hexdigest()
if fp in seen: continue
seen.add(fp)
# + Use INSERT OR IGNORE in SQLite for cross-run dedup
```

### SQLite Pattern
```python
conn = sqlite3.connect('data.db')
conn.execute('CREATE TABLE IF NOT EXISTS items (title TEXT UNIQUE, price REAL)')
conn.execute('INSERT OR IGNORE INTO items VALUES (?,?)', (title, price))
conn.commit(); conn.close()
```

### Logging
```python
logging.basicConfig(level=logging.INFO,
    format='%(asctime)s | %(levelname)s | %(message)s',
    handlers=[logging.FileHandler('scraper.log'), logging.StreamHandler()])
logger = logging.getLogger('scraper')
logger.info('Fetched page'); logger.error('Failed!')
```

### Polite Scraping (always!)
```python
time.sleep(random.uniform(1, 3))    # Between pages
time.sleep(random.uniform(0.3, 1))  # Between items
```

### ScrapingPipeline Class — Use This Template
```python
class ScrapingPipeline:
    def __init__(self, db_file):  self._init_db(); self.seen = set()
    def fetch(self, url):         return fetch_with_retry(url)
    def parse(self, soup):        # extract + clean + dedup
    def save(self, items):        # INSERT OR IGNORE to SQLite
    def run(self, urls):          # loop fetch → parse → save → sleep
```

---
# 🎓 Course Complete — Full Summary

## What You've Built Across 3 Notebooks

```
NOTEBOOK 1 — requests + BeautifulSoup4
  ✅ requests.get()         Fetch any webpage
  ✅ BeautifulSoup          Parse HTML into a searchable tree
  ✅ find() / find_all()    Search by tag, class, id, regex
  ✅ CSS Selectors          select() / select_one()
  ✅ Text cleaning          strip(), get_text(), type conversion
  ✅ Headers / User-Agent   Avoid getting blocked
  ✅ Error handling         try/except, raise_for_status
  ✅ Inspect Element        Find any tag on any real website
  ✅ None safety            Never crash on missing tags
  ✅ Tables → dicts         th, tr, td pipeline
  ✅ CSV / JSON             Save scraped data
  ✅ time.sleep             Rate limiting
  ✅ Pagination             Next-button loop
  ✅ Projects 1-3           Quotes, Books, Full paginated scraper

NOTEBOOK 2 — Selenium
  ✅ When to use Selenium   JS-rendered pages
  ✅ WebDriver + headless   get_driver() helper
  ✅ find_element()         By.CSS_SELECTOR, ID, CLASS
  ✅ WebDriverWait          Proper waits (not time.sleep!)
  ✅ click() / send_keys()  Interact with buttons and forms
  ✅ Dropdowns              Select class
  ✅ Alerts / iframes       switch_to.alert / switch_to.frame
  ✅ Scrolling              execute_script() + infinite scroll
  ✅ Screenshots            Debug what Selenium sees
  ✅ Anti-detection         Remove webdriver flag, random delays
  ✅ Selenium + BS4         Best combo pattern
  ✅ Projects 4-5           JS scraper + Login + protected page

NOTEBOOK 3 — Production Pipelines
  ✅ Smart fetcher          requests → Selenium fallback auto
  ✅ Reusable cleaners      safe_text, to_float, to_bool
  ✅ Retry + backoff        Exponential backoff, 404 handling
  ✅ Logging                File + terminal, log levels
  ✅ SQLite storage         CREATE, INSERT OR IGNORE, query
  ✅ Deduplication          Fingerprinting + seen set
  ✅ ScrapingPipeline class All pieces in one reusable class
  ✅ Project 6 — Jobs       Title, company, location → SQLite + CSV
  ✅ Project 7 — Prices     Track changes over time + alerts
```

---

## 🎯 What To Do Next

### Practice Projects (try on your own!)

| Project | Site | Tools |
|---------|------|-------|
| Scrape a Wikipedia table | wikipedia.org | requests + BS4 |
| Top IMDb movies | imdb.com | Selenium + BS4 |
| GitHub trending repos | github.com/trending | requests + BS4 |
| Myntra product prices | myntra.com | Selenium (JS page) |
| Indeed job listings | indeed.com | Selenium + Pipeline |

### For MAANG Interviews — Focus On
```
1. Explain the difference between requests and Selenium
2. How do you handle anti-scraping measures?
3. How do you make a scraper production-ready?
4. How do you store and deduplicate scraped data?
5. What is exponential backoff and why use it?
```

### Recommended Next Steps
```
→ Scrapy       (async scraping framework, for large-scale)
→ Playwright   (modern Selenium alternative, faster)
→ PostgreSQL   (production database instead of SQLite)
→ Airflow      (schedule scrapers to run automatically)
→ Docker       (package your scraper to run anywhere)
```

---

> **You started at zero. You can now build a production-grade web scraper.**
> Keep building. Keep scraping. 🚀